In [1]:
import pandas as pd

df_maps = pd.read_csv("../data_samples/sample_10k_first_keywords_attemp2.csv")

df_maps.head()

,id,uid,url,url_status,text,original_width,original_height,clip_l14_similarity_score,sha256,score,pos,neg,score_1,pos_1,neg_1
0,0,337d71949d50f8f2d8805a8b3fa3d81b,http://a0.fast-meteo.com/maps/static/t_Austral...,ERR,Australia Nuvola,120,96.0,0.259766,e834f3dd906b36869fd707537734b6f7dfce2f2dcfda0d...,0.0,[],[],0.0,[],[]
1,1,eae4819aef3d65027b8d28a230afc0f9,https://cdn1.vectorstock.com/i/thumbs/22/75/fr...,OK,French Southern Territories Rubber Stamps vect...,112,118.0,0.362793,2ca3b7f2d2f2996a3800c001797ed3c7ee666ca82dbb87...,-5.0,[],"['stamps', 'image']",-5.0,[],"['stamps', 'image']"
2,2,4ea106e4a5867df0f987300f2038cd4c,http://7.hikb.at:80/img/iconmenu/attraction.png,OK,New York - Látnivalók,64,64.0,0.127808,c3f92275db5f9b35f2efe57d85fc0f547347b1fd361646...,0.0,[],[],0.0,[],[]
3,3,73a0bd1cc6080dd8985e2b790fe9a113,https://aa1cc5ce659c48a883597af217799a86.objec...,ERR,Karte mit 5 bettel,210,210.0,0.227539,4a46099a1e18fefd2523a532f4f3c0563926a90496bd91...,0.0,[],[],0.0,[],[]
4,4,030e8a94dd1934c8c20eb85cc04d009d,http://images.slideplayer.com/9/2524694/slides...,OK,2004 Weekday Arterial Hourly Volumes: 6 p.m. t...,960,720.0,0.451660,98e4271537a8193cef6ba4a5f3f64ba60f49d11aa19972...,0.0,[],[],0.0,[],[]


In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np

class MapViewerBatch:
    def __init__(self, df, sort_by_score=None, batch_size=4, show_text=True):
        self.df = df.copy().reset_index(drop=True)
        self.batch_size = batch_size
        self.show_text = show_text

        if sort_by_score == 'asc':
            self.df.sort_values('score_1', ascending=True, inplace=True)
        elif sort_by_score == 'desc':
            self.df.sort_values('score_1', ascending=False, inplace=True)

        self.indices = np.arange(len(self.df))
        self.current_position = 0

        self.image_widget = widgets.Output()
        self.prev_button_top = widgets.Button(description='← Previous')
        self.next_button_top = widgets.Button(description='Next →')
        self.prev_button_bottom = widgets.Button(description='← Previous')
        self.next_button_bottom = widgets.Button(description='Next →')
        self.counter_widget = widgets.HTML()

        for btn in [self.prev_button_top, self.prev_button_bottom]:
            btn.on_click(self.show_previous)
        for btn in [self.next_button_top, self.next_button_bottom]:
            btn.on_click(self.show_next)

        self.nav_top = widgets.HBox([self.prev_button_top, self.counter_widget, self.next_button_top])
        self.nav_bottom = widgets.HBox([self.prev_button_bottom, self.next_button_bottom])

        self.container = widgets.VBox([self.nav_top, self.image_widget, self.nav_bottom])

    def get_current_indices(self):
        start = self.current_position
        end = min(self.current_position + self.batch_size, len(self.df))
        return self.indices[start:end]

    def show_images(self, b=None):
        self.image_widget.clear_output(wait=True)

        with self.image_widget:
            for idx in self.get_current_indices():
                row = self.df.iloc[idx]

                if not self.show_text:
                    info_html = f"""
                <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                    <div style='flex:1; padding-right:10px;'>
                        <b>Score:</b> {row['score_1']}<br>
                        <b>Positive keywords:</b> {row['pos_1'] if row['pos_1'] else 'None'}<br>
                        <b>Negative keywords:</b> {row['neg_1'] if row['neg_1'] else 'None'}<br>
                        <b>Text-image similarity:</b> {row['clip_l14_similarity_score']}</br>
                    </div>
                    <div>
                        <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                    </div>
                </div>
                """
                else:
                    info_html = f"""
                    <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                        <div style='flex:1; padding-right:10px;'>
                            <b>Text:</b> {row['text']}<br>
                            <b>Score:</b> {row['score_1']}<br>
                            <b>Positive keywords:</b> {row['pos_1'] if row['pos_1'] else 'None'}<br>
                            <b>Negative keywords:</b> {row['neg_1'] if row['neg_1'] else 'None'}<br>
                            <b>Text-image similarity:</b> {row['clip_l14_similarity_score']}</br>
                        </div>
                        <div>
                            <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                        </div>
                    </div>
                    """
                display(HTML(info_html))


            self.counter_widget.value = f"<center><b>Mapy {self.current_position + 1} – {min(self.current_position + self.batch_size, len(self.df))} / {len(self.df)}</b></center>"


    def show_next(self, b=None):
        if self.current_position + self.batch_size < len(self.indices):
            self.current_position += self.batch_size
            self.show_images()

    def show_previous(self, b=None):
        if self.current_position - self.batch_size >= 0:
            self.current_position -= self.batch_size
            self.show_images()

    def display(self):
        self.show_images()
        display(self.container)


In [3]:
positive_df = df_maps[df_maps['score_1'] > 0]
negative_df = df_maps[df_maps['score_1'] < 0]
neutral_df = df_maps[df_maps['score_1'] == 0]

In [6]:
MapViewerBatch(positive_df, sort_by_score='desc', batch_size=10, show_text=True).display()

### Sprawdzenie słownika na czystym zbiorze

In [1]:
from datasets import load_dataset

dataset = load_dataset("sraimund/MapPool", split="train", streaming=True)

Resolving data files:   0%|          | 0/23876 [00:00<?, ?it/s]

In [2]:
sample_100k = list(dataset.take(100000))

In [3]:
from tqdm import tqdm

data = []
for item in tqdm(sample_100k, desc="Przetwarzanie map"):
    row = {
        'uid': item.get('uid'),
        'url': item.get('url'),
        'text': item.get('text'),
        'original_width': item.get('original_width'),
        'original_height': item.get('original_height'),
        'clip_l14_similarity_score': item.get('clip_l14_similarity_score'),
        'sha256': item.get('sha256'),
        'l14_img': item.get('l14_img'),
    }

    data.append(row)

df = pd.DataFrame(data)

print(f"\n✓ Utworzono DataFrame: {len(df)} wierszy, {len(df.columns)} kolumn")

Przetwarzanie map: 100%|██████████| 100000/100000 [00:05<00:00, 19580.40it/s]



✓ Utworzono DataFrame: 100000 wierszy, 8 kolumn


In [4]:
import json

with open("./keywords/positive_keywords.json", "r", encoding="utf-8") as f:
    positive_json = json.load(f)

with open("./keywords/negative_keywords.json", "r", encoding="utf-8") as f:
    negative_json = json.load(f)


print(f"Loaded {len(positive_json)} positive keywords.")
print(f"Loaded {len(negative_json)} negative keywords.")

Loaded 2725 positive keywords.
Loaded 2415 negative keywords.


In [5]:
import ahocorasick

A = ahocorasick.Automaton()

for phrase, value in positive_json.items():
    A.add_word(phrase.lower(), ("pos", phrase, value))

for phrase, value in negative_json.items():
    A.add_word(phrase.lower(), ("neg", phrase, value))

A.make_automaton()

In [6]:
def compute_score_full_words(text):
    if not isinstance(text, str):
        text = str(text)

    text_lower = text.lower()
    score = 0
    used_phrases = set()
    pos_keys = []
    neg_keys = []

    for end_index, (ptype, phrase, value) in A.iter(text_lower):
        start_index = end_index - len(phrase) + 1

        if (start_index == 0 or not text_lower[start_index-1].isalnum()) and \
           (end_index == len(text_lower)-1 or not text_lower[end_index+1].isalnum()):

            if phrase not in used_phrases:
                score += value
                used_phrases.add(phrase)
                if ptype == "pos":
                    pos_keys.append(phrase)
                else:
                    neg_keys.append(phrase)

    return score, pos_keys, neg_keys

In [9]:
start = pd.Timestamp.now()
df[["score", "pos", "neg"]] = df["text"].apply(
    lambda x: pd.Series(compute_score_full_words(x))
)
end = pd.Timestamp.now()
print(f"✓ Obliczono wyniki dla {len(df)} wierszy w czasie {end - start}")

✓ Obliczono wyniki dla 100000 wierszy w czasie 0 days 00:00:21.305547


In [10]:
df[["text", "url", "score", "pos", "neg"]]

,text,url,score,pos,neg
0,5.5 ha antes de pueblo edén - nyp37745p,https://www.medanos.com.uy/f/192/3/5/1/0/0/d67...,0.0,[],[]
1,World maps mappery world outline map gumiabroncs,http://www.mappery.com/maps/World-Outline-Map....,0.0,[],[]
2,no Photo,http://as.rdcpix.com/776088671/19fadb255d5df87...,-3.0,[],[photo]
3,Via Papa Giovanni Paolo II,https://maps.googleapis.com/maps/api/staticmap...,0.0,[],[]
4,Stopy na běžky - od hřbitova směr Březina dole...,https://hradeczije.cz/wp-content/uploads/2017/...,0.0,[],[]
...,...,...,...,...,...
99995,Landing Page Map,https://maps.googleapis.com/maps/api/staticmap...,0.0,[],[]
99996,"Foto de terreno habitacional en venta en , pl...",https://propiedadescom.s3.amazonaws.com/files/...,-5.0,[],"[foto, calles]"
99997,Chrysler Town &amp; Country Questions,https://static.cargurus.com/images/site/2015/0...,0.5,[country],[town]
99998,First Floor Plan of Mediterranean Ranch House ...,https://i.pinimg.com/236x/39/a3/ee/39a3ee10b90...,-14.0,[],"[floor plan, plan, layout, plans, floor plans]"


In [11]:
df_maps_positive = df[df['score'] > 0].reset_index(drop=True)
df_maps_negative = df[df['score'] < 0].reset_index(drop=True)
df_maps_neutral = df[df['score'] == 0].reset_index(drop=True)

print(f"Positive maps: {len(df_maps_positive)}")
print(f"Negative maps: {len(df_maps_negative)}")
print(f"Neutral maps: {len(df_maps_neutral)}")

Positive maps: 6773
Negative maps: 32561
Neutral maps: 60666


In [12]:
output_df = df[['uid', 'url', 'text', 'score', 'pos', 'neg']]

output_df.to_csv("../data/sample_100k_first_keywords_full_words.csv", index=False)

In [13]:
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np

class MapViewerBatch:
    def __init__(self, df, sort_by_score=None, batch_size=4, show_text=True):
        self.df = df.copy().reset_index(drop=True)
        self.batch_size = batch_size
        self.show_text = show_text

        if sort_by_score == 'asc':
            self.df.sort_values('score', ascending=True, inplace=True)
        elif sort_by_score == 'desc':
            self.df.sort_values('score', ascending=False, inplace=True)

        self.indices = np.arange(len(self.df))
        self.current_position = 0

        self.image_widget = widgets.Output()
        self.prev_button_top = widgets.Button(description='← Previous')
        self.next_button_top = widgets.Button(description='Next →')
        self.prev_button_bottom = widgets.Button(description='← Previous')
        self.next_button_bottom = widgets.Button(description='Next →')
        self.counter_widget = widgets.HTML()

        for btn in [self.prev_button_top, self.prev_button_bottom]:
            btn.on_click(self.show_previous)
        for btn in [self.next_button_top, self.next_button_bottom]:
            btn.on_click(self.show_next)

        self.nav_top = widgets.HBox([self.prev_button_top, self.counter_widget, self.next_button_top])
        self.nav_bottom = widgets.HBox([self.prev_button_bottom, self.next_button_bottom])

        self.container = widgets.VBox([self.nav_top, self.image_widget, self.nav_bottom])

    def get_current_indices(self):
        start = self.current_position
        end = min(self.current_position + self.batch_size, len(self.df))
        return self.indices[start:end]

    def show_images(self, b=None):
        self.image_widget.clear_output(wait=True)

        with self.image_widget:
            for idx in self.get_current_indices():
                row = self.df.iloc[idx]

                if not self.show_text:
                    info_html = f"""
                <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                    <div style='flex:1; padding-right:10px;'>
                        <b>Score:</b> {row['score']}<br>
                        <b>Positive keywords:</b> {', '.join(row['pos']) if row['pos'] else 'None'}<br>
                        <b>Negative keywords:</b> {', '.join(row['neg']) if row['neg'] else 'None'}<br>
                    </div>
                    <div>
                        <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                    </div>
                </div>
                """
                else:
                    info_html = f"""
                    <div style='display:flex; align-items:flex-start; padding:5px; margin-bottom:15px; border:1px solid #eee; border-radius:5px;'>
                        <div style='flex:1; padding-right:10px;'>
                            <b>Text:</b> {row['text']}<br>
                            <b>Score:</b> {row['score']}<br>
                            <b>Positive keywords:</b> {', '.join(row['pos']) if row['pos'] else 'None'}<br>
                            <b>Negative keywords:</b> {', '.join(row['neg']) if row['neg'] else 'None'}<br>
                        </div>
                        <div>
                            <img src="{row["url"]}" style="max-width:300px; border:2px solid #ddd; border-radius:5px;">
                        </div>
                    </div>
                    """
                display(HTML(info_html))


            self.counter_widget.value = f"<center><b>Mapy {self.current_position + 1} – {min(self.current_position + self.batch_size, len(self.df))} / {len(self.df)}</b></center>"


    def show_next(self, b=None):
        if self.current_position + self.batch_size < len(self.indices):
            self.current_position += self.batch_size
            self.show_images()

    def show_previous(self, b=None):
        if self.current_position - self.batch_size >= 0:
            self.current_position -= self.batch_size
            self.show_images()

    def display(self):
        self.show_images()
        display(self.container)


In [18]:
MapViewerBatch(df_maps_negative, sort_by_score='desc', batch_size=10, show_text=True).display()